# Creates cell masks by taking the CAAX channel, filling in holes, and removing unconnected objects

### Most useful for cells that are clumped together (hard to separate using the cell channel) with a strong CAAX+ cell border

In [ ]:
from pathlib import Path

from bioio import BioImage
import bioio_ome_tiff
from bioio.writers import OmeTiffWriter
import bioio_tifffile

import numpy as np
import pandas as pd

%matplotlib notebook
%matplotlib inline
import matplotlib.pyplot as plt

import sys
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
import src.d00_utils.utilities as utils
import src.d00_utils.dirnames as dn
from src.d01_init_proc import vis_and_rescale
from src.d01_init_proc import subtractbg

from scipy import ndimage as ndi
from skimage.measure import label, regionprops

from skimage.filters import threshold_otsu, threshold_multiotsu
from skimage import morphology
import numpy.ma as ma

%load_ext autoreload
#%autoreload 2

In [ ]:
imgpath = Path('/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab Box/Z-lab shared folders/Kathryn + Eduardo/CE031/experiment/img_processing/caax_cell_stack/CE031_div3_I1-A1_scP2_aligned.ome.tif')
img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
physical_pixel_sizes = img_file.physical_pixel_sizes
dim_order = img_file.dims.order

In [ ]:
mask_ch = -1
#mask_dirpath = Path(input())

mask_dirpath = Path('/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab Box/Z-lab shared folders/Kathryn + Eduardo/CE031/experiment/img_processing/masks/refinedmasks')

In [ ]:
masked_dirpath = utils.get_proc_dirpath(mask_dirpath) / dn.masks_dirname / 'masked_cell_caax'
masked_dirpath.mkdir(exist_ok=True)

In [ ]:
maskpaths = [Path(imgpath) for imgpath in mask_dirpath.glob('*.tif')]
maskpaths.sort()

for i, maskpath in enumerate(maskpaths):
    maskname = maskpath.name
    print(f'Processing {maskname}: {i}/{len(maskpaths)}')

    mask = BioImage(maskpath, reader=bioio_tifffile.Reader).data
    img = mask[:, [0, 1], :, :, :]
    bin_mask = (mask[:, mask_ch, np.newaxis, :, :, :] > 0).astype('int')
    mask_exp = np.broadcast_to(bin_mask, img.shape)
    masked_img = (img * mask_exp).astype('uint16')

    ome_metadata = OmeTiffWriter.build_ome(data_shapes=[masked_img.shape], data_types=[masked_img.dtype], dimension_order=[dim_order],
                                           physical_pixel_sizes=[physical_pixel_sizes])
    OmeTiffWriter.save(masked_img, masked_dirpath / maskname, ome_xml = ome_metadata)


In [ ]:
imgpath = Path('/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab Box/Z-lab shared folders/Kathryn + Eduardo/CE031/experiment/img_processing/masks/masked_cell_caax/CE031_div3_I1-A1_scP1_aligned_ROI2.ome.tif')

In [ ]:
img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
img = img_file.data

In [ ]:
ch = 1
img_chsubset = img[:, ch, np.newaxis, :, :, :]

In [ ]:
outlier_percs = [10, 20, 30, 40, 50]
preproc = []
for op in outlier_percs:
    img_preprocessed = subtractbg.clip_upper_outliers(img_chsubset, outlier_perc=op)
    preproc.append(img_preprocessed)
vis_and_rescale.create_fig(preproc, outlier_percs)



In [ ]:
from scipy.ndimage import gaussian_filter

sigma = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
preproc = []
for i in sigma:
    img_preprocessed = gaussian_filter(img_chsubset, sigma=i)
    preproc.append(img_preprocessed)
vis_and_rescale.create_fig(preproc, sigma)

In [ ]:
bgsbimgs = []

sigma = [0, 0.1]
preproc = []
img_preprocessed = subtractbg.clip_upper_outliers(img_chsubset, outlier_perc=30)
for i in sigma:
    zeros = np.where(img_preprocessed==0)
    img_preproc = gaussian_filter(img_preprocessed, sigma=i)
    img_preproc[zeros] = 0
    otsu_thresholds = subtractbg.get_otsu_thresholds(img_preproc)
    bgsbimg = subtractbg.subtract_background(img_chsubset, otsu_thresholds)

    bin_bgsb = (bgsbimg > 0) * 255
    print(np.squeeze(otsu_thresholds))
    bgsbimgs.append(bin_bgsb)
vis_and_rescale.create_fig(bgsbimgs, sigma, show=False)

In [ ]:
edited = morphology.binary_opening(bgsbimg)
edited = morphology.binary_closing(edited)
edited = morphology.binary_opening(bgsbimg)
edited = morphology.binary_closing(edited)
edited = morphology.binary_opening(bgsbimg)
edited = morphology.binary_closing(edited)
edited = morphology.binary_opening(bgsbimg)
edited = morphology.binary_closing(edited)
edited = morphology.binary_opening(bgsbimg)
edited = morphology.binary_closing(edited)
edited = morphology.binary_opening(bgsbimg)
edited = morphology.binary_closing(edited)

e2 = ndi.binary_fill_holes(edited, 1000)
e2 = morphology.remove_small_holes(e2, 1000)

fig = vis_and_rescale.create_fig([bgsbimg > 0, edited, e2], ['bgsbimg', 'binary', 'e2'])

In [ ]:
a = morphology.remove_small_holes(e2, 1000)

e2 = morphology.remove_small_objects(e2, 1000)

In [ ]:
diff = (a!=e2).astype('int')
print(np.sum(diff))